<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
Was sind KI-Agenten? Tool Use & Function Calling
</b></font> </br>

---

**Leitprojekt:** Der gesamte Kurs baut auf einem durchgehenden Anwendungsfall auf — dem **Meeting- & Research-Briefing-Agent**, einem kontrollierten Agentensystem, das Projektunterlagen und Meeting-Protokolle sichtet, Aussagen mit Quellen belegt und bei Unsicherheit eskaliert.



**Beitrag zum Leitprojekt:** M01 legt die begriffliche Grundlage und die erste kontrollierte Handlungsebene. Die Progression von einfachem zu kooperierendem Agenten (siehe Timeline unten) ist genau der Weg, den der Meeting- & Research-Briefing-Agent im Kursverlauf durchläuft — von "Such-Tool mit nachvollziehbaren Entscheidungen" bis zu "HITL, Memory und spezialisierten Workern". Bevor der Agent Protokolle und Unterlagen sichtet, muss er planen, welche Fähigkeiten ihm zur Verfügung stehen, und anschließend gezielt handeln — die Tool-Schnittstelle macht sichtbar, was erlaubt, beobachtbar und prüfbar ist.

In [ ]:
#@title  🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os

os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "M01-KI-Agenten-und-Tool-Use"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)

print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS
# LangSmith Tracing
run_cfg = {
    "run_name": "M01_KI_Agenten_und_Tool_Use",
    "tags": ["m01", "ki-agenten", "tool-use"],
    "metadata": {"notebook": "M01", "version": "1.0"}
}

# 1 | Was ist ein KI-Agent?
---

Wenn man sich mit generativer KI beschäftigt, stößt man früher oder später auf den Begriff **Agent** – also ein System, das Aufgaben eigenständig ausführt. Doch was genau ist ein *„echter“* Agent? Muss er vollständig autonom sein?



<img src="https://raw.githubusercontent.com/ralf-42/Agenten/main/07_image/agentisch_2.png" class="logo" width="900"/>
<p><font color='black' size="2">
KI-generiertes Bild
</font></p>




Besonders einfach ist der Einstieg bei eher einfachen Aufgaben – also bei Prozessen, die heute noch manuell erledigt werden, wie das Ausfüllen von Formularen, das Nachschlagen in einer Datenbank oder das Kopieren von Informationen zwischen verschiedenen Systemen. Diese Aufgaben lassen sich gut in sogenannte agentische Workflows überführen – also in Abläufe, bei denen die KI (teilweise) selbstständig handelt.


Natürlich gibt es auch deutlich komplexere Anwendungen, bei denen die KI viele Entscheidungen trifft, Schleifen durchläuft und sich an neue Situationen anpasst. Solche Systeme sind spannend – aber gerade für den Anfang ist es oft sinnvoller, mit kleineren, überschaubaren Schritten zu starten. Dort liegen aktuell auch die meisten Chancen, GenAI im Alltag oder im Beruf sinnvoll einzusetzen.



Generative KI und KI-Agenten bauen aufeinander auf, sind aber nicht dasselbe.

| Ebene | Beschreibung | Typisches Ergebnis |
|---|---|---|
| **LLM-Call** | Ein Prompt, eine Antwort | Textantwort |
| **GenAI-App** | Prompt + UI + einfache Logik | Chatbot, Assistent |
| **KI-Agent** | LLM + Tool-Nutzung + Entscheidungslogik + Schleife | Mehrstufige Aufgabenerfüllung |




**Wann reicht ein LLM-Call?**
- Reines Umformulieren, Zusammenfassen, Übersetzen
- Keine externen Datenquellen oder Aktionen nötig

**Wann lohnt ein Agent?**
- Mehrere Schritte mit Zwischenentscheidungen
- Externe Aktionen (APIs, Datenbanken, Dateien)
- Rückfragen, Fehlerbehandlung und Zielverfolgung

# 2 | Die 5 Eigenschaften eines Agenten
---

Ein praxisreifer Agent zeigt fünf Kerneigenschaften:

1. **Autonomie**: kann Teilentscheidungen selbst treffen.

2. **Reaktionsfähigkeit**: reagiert auf neue Informationen.

3. **Proaktivität**: verfolgt ein Ziel über mehrere Schritte.

4. ***Soziale*** **Fähigkeit**: interagiert klar mit Mensch und/oder anderen Agenten.

5. **Handlungsfähigkeit**: kann Werkzeuge, APIs oder Systeme aktiv einsetzen, um Schritte umzusetzen.

Je mehr davon robust umgesetzt sind, desto eher sprechen wir von einem Agenten statt einem einfachen Chat-Interface.

# 3 | ReAct-Prinzip
---

Das ReAct-Prinzip (**T**hought → **A**ction → **O**bservation) ist der mentale Grundzyklus agentischer Systeme:

- **Reasoning/Thought**: nächsten sinnvollen Schritt planen
- **Action**: Tool, Datenquelle oder Modellaufruf nutzen
- **Observation**: Ergebnis prüfen und den nächsten Schritt ableiten

Das ReAct-Prinzip ist ein zentrales mentales Grundmuster vieler agentischer Systeme"

# 4 | Was sind Tools?
---

Tools sind Funktionen, die ein KI-Agent gezielt aufrufen kann – für Aufgaben, die das Sprachmodell allein nicht lösen kann: aktuelle Daten abfragen, rechnen, externe Systeme ansprechen.


Ein **Tool** ist eine Funktion, die ein Agent während seines ReAct-Zyklus aufrufen kann.

| Eigenschaft | Beschreibung |
|---|---|
| **Name** | Eindeutige Bezeichnung (= Funktionsname) |
| **Beschreibung** | Docstring – erklärt dem LLM, *wann* das Tool passt |
| **Schema** | Automatisch aus Typ-Hints generiert (JSON Schema) |
| **Rückgabewert** | String oder serialisierbarer Python-Typ |


Das LLM sieht **nur das Schema** (Name + Beschreibung + Parameter-Typen), niemals den eigentlichen Code. Damit ist das Schema ein **Agenten-Vertrag**: eine feste Schnittstelle, auf die sich Agent und Tool verlassen – ändert sich der Vertrag (z. B. ein Parametertyp), muss der Agent das zuverlässig erkennen können, statt am fehlerhaften Aufruf zu scheitern.

<p><font color='darkblue' size="4">💡 <b>Tipp</b></font></p>

Je klarer der Docstring, desto besser wählt der Agent das richtige Tool.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Kommunikationsdiagramm</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
sequenceDiagram
    autonumber
    participant U as Benutzer
    participant A as Agent (LLM)
    participant T as Tool

    U->>A: "Wie warm ist 37°C in Fahrenheit?"
    A->>A: Prüft verfügbare Tool-Schemas
    A->>T: celsius_nach_fahrenheit(temperatur=37.0)
    T->>T: Umrechnung °C auf °F
    T->>A: 98.6
    A->>U: "37 °C entsprechen 98,6° F"
'''

mermaid(diagram, width=750)

# 5 | @tool Decorator
---


Der `@tool`-Decorator aus `langchain_core.tools` verwandelt eine normale Python-Funktion in ein LangChain-Tool.

**Was passiert dabei automatisch:**
1. **Name** wird aus dem Funktionsnamen abgeleitet
2. **Beschreibung** wird aus dem Docstring übernommen
3. **Schema** wird aus den Typ-Hints als JSON Schema generiert

**Pflichtangaben für gute Tools:**
- ✅ Typ-Hints für alle Parameter und den Rückgabewert
- ✅ Klarer Docstring: was macht das Tool, was gibt es zurück?
- ✅ Sprechende Parameternamen

Das generierte Schema ist exakt das, was das LLM sieht, wenn es entscheidet, ob und wie es das Tool aufruft.

In [ ]:
import json
from langchain_core.tools import tool

@tool
def celsius_nach_fahrenheit(temperatur: float) -> float:
    """Rechnet eine Temperatur von Celsius in Fahrenheit um. Gibt den Fahrenheit-Wert zurück."""
    return round(temperatur * 9 / 5 + 32, 2)

# Schema anzeigen – was der Agent "sieht"
mprint("## Tool-Schema (was das LLM sieht)")
mprint(f"**Name:** `{celsius_nach_fahrenheit.name}`")
mprint(f"**Beschreibung:** {celsius_nach_fahrenheit.description}")
schema_json = json.dumps(
    celsius_nach_fahrenheit.args_schema.model_json_schema(),
    indent=2,
    ensure_ascii=False
)
mprint(f"**Parameter-Schema:**\n```json\n{schema_json}\n```")

# 6 | Weiteres Tool bauen
---


Zwei praxisnahe Tools werden isoliert getestet – ohne Agent.

**Tipp:** Tools immer zuerst isoliert testen, bevor diese in einen Agenten eingebunden werden. Das erleichtert das Debugging erheblich.

In [ ]:
from langchain_core.tools import tool

@tool
def ist_buerozeit(stunde: int) -> str:
    """Prüft ob eine Stunde des Tages (0–23) in die typische Bürozeit fällt (9–17 Uhr).
    Gibt 'Bürozeit' oder 'Außerhalb Bürozeit' zurück."""
    if 9 <= stunde <= 17:
        return "Bürozeit"
    return "Außerhalb Bürozeit"

# Alle verfügbaren Tools auflisten
tools = [celsius_nach_fahrenheit, ist_buerozeit]
mprint(f"**Definierte Tools:** `{[t.name for t in tools]}`")

In [ ]:
# .func() ruft die Python-Funktion direkt auf – kein Runnable, kein Tracing
mprint("## Tool-Tests")
mprint('---')
mprint(f"**37 °C → Fahrenheit:** {celsius_nach_fahrenheit.func(temperatur=37.0)} °F")
mprint(f"**100 °C → Fahrenheit:** {celsius_nach_fahrenheit.func(temperatur=100.0)} °F")
mprint(f"**0 °C → Fahrenheit:** {celsius_nach_fahrenheit.func(temperatur=0.0)} °F")
print()
mprint(f"**14 Uhr – Bürozeit?** {ist_buerozeit.func(stunde=14)}")
mprint(f"**20 Uhr – Bürozeit?** {ist_buerozeit.func(stunde=20)}")
mprint(f"**9 Uhr – Bürozeit?** {ist_buerozeit.func(stunde=9)}")

# 7 | Tool mit Fehlerbehandlung
---


Tools dürfen **keine ungefangenen Exceptions werfen** – das würde den Agent-Loop unterbrechen.

**Grundregeln für robuste Tools:**
- ✅ Eingaben validieren (Typen, Wertebereiche, erlaubte Werte)
- ✅ Fehler als informativen String zurückgeben, nicht als Exception
- ✅ Fehlermeldungen sollen dem LLM helfen, die Situation zu verstehen

Das LLM kann aus einer klaren Fehlermeldung ("Währung CHF nicht unterstützt, verfügbar: EUR, USD, GBP") selbst entscheiden, wie es weiter vorgeht.

In [ ]:
from langchain_core.tools import tool

@tool
def waehrung_umrechnen(betrag: float, von: str, nach: str) -> str:
    """Rechnet einen Geldbetrag zwischen EUR, USD und GBP um.
    Unterstützte Währungen: EUR, USD, GBP.
    Gibt das Ergebnis als formatierten String zurück."""
    kurse = {
        ("EUR", "USD"): 1.08, ("USD", "EUR"): 0.93,
        ("EUR", "GBP"): 0.85, ("GBP", "EUR"): 1.18,
        ("USD", "GBP"): 0.79, ("GBP", "USD"): 1.27,
    }
    von_norm = von.upper().strip()
    nach_norm = nach.upper().strip()

    if von_norm == nach_norm:
        return f"{betrag:.2f} {nach_norm} (keine Umrechnung nötig)"

    kurs = kurse.get((von_norm, nach_norm))
    if kurs is None:
        return (
            f"Fehler: Umrechnung von {von_norm} nach {nach_norm} nicht unterstützt. "
            f"Verfügbare Währungen: EUR, USD, GBP."
        )

    ergebnis = round(betrag * kurs, 2)
    return f"{betrag:.2f} {von_norm} = {ergebnis:.2f} {nach_norm} (Kurs: {kurs})"

In [ ]:
# .func() ruft die Python-Funktion direkt auf – kein Runnable, kein Tracing
mprint("## Fehlerbehandlung testen")
mprint(f"✅ Normal:              {waehrung_umrechnen.func(betrag=100.0, von='EUR', nach='USD')}")
mprint(f"✅ Gleiche Währung:     {waehrung_umrechnen.func(betrag=50.0, von='USD', nach='USD')}")
mprint(f"⚠️ Unbekannte Währung:  {waehrung_umrechnen.func(betrag=100.0, von='EUR', nach='CHF')}")

# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Geeignet ist sie zum Beispiel, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


<p><font color='darkgreen' size="4">
📋 <b>Quick-Template für Übungsaufgaben</b>
</font></p>

Für alle Übungsaufgaben steht ein **Quick-Template** zur Verfügung:

> **`Quick-Template.ipynb`** — enthält die Standard-Umgebung (Setup, API-Keys, Imports) als fertigen Startpunkt. Einfach öffnen, kopieren und loslegen — kein Boilerplate von Grund auf neu schreiben.

**Tipp für Google Colab:**
> In Google Colab steht **Gemini** direkt als KI-Assistent zur Verfügung (Seitenleiste → ✨). Ideal zum Erkunden von Code-Varianten, Verstehen von Fehlermeldungen oder als Ideengeber für eigene Lösungsansätze.

**Grundlagen**
- Einen realen Prozess aus dem Briefing-Agent-Kontext oder dem eigenen Arbeitskontext beschreiben.
- Begründet entscheiden: Reicht ein LLM-Call, reicht ein Chatbot oder ist ein KI-Agent nötig?


--- Text ---

**Aufbau**
- Den ReAt/TAO-Zyklus für den Use Case in 3-5 Schritten beschreiben.
- Die Stellen benennen, an denen der Agent denken, handeln und beobachten muss.


--- Text ---

**Vertiefung**
- 2 konkrete Tools definieren, die der Agent benötigt.
- Kurz begründen, warum diese Tools besser als ein reiner Prompt ausreichen.


--- Text ---

---

**Teil 2 — Tools bauen**

<p><font color='black' size="5">
Tools für den Meeting-Briefing-Agent
</font></p>

Der Meeting-Briefing-Agent benötigt kleine, klar beschriebene Werkzeuge: Begriffe normalisieren, Suchanfragen vorbereiten und Metadaten auswerten. Dieser Abschnitt legt dafür die Tool-Grundlage.

**Grundlagen**
- Mindestens zwei Tools mit dem `@tool`-Decorator definieren.
- Alle Parameter und Rückgabewerte mit Typ-Hints versehen.
- Tools direkt mit `.func(...)` testen.
- Tools in `meine_tools` speichern.

**✅ Erledigt wenn:** `meine_tools` enthält mindestens zwei Tools mit Namen, Docstring und Args-Schema.


In [ ]:
# Grundlagen: Tools für einfache Briefing-Agent-Hilfsfunktionen
from langchain_core.tools import tool

# 1. normalisiere_suchbegriff(begriff: str) -> str: Kleinschreibung + Whitespace normalisieren
# 2. zaehle_suchbegriffe(frage: str) -> int: Woerter > 2 Zeichen zaehlen
# 3. Beide Tools in meine_tools sammeln, mit .func(...) testen

**Aufbau**
- Mindestens ein Tool mit Fehlerbehandlung ergänzen.
- Ungültige Eingaben als verständlichen Fehlerstring zurückgeben.
- Den Fehlerfall direkt mit `.func(...)` testen.

**✅ Erledigt wenn:** Ein Fehlerfall liefert einen String mit `Fehler` statt einer Exception.


In [ ]:
# Aufbau: Fehlerbehandlung in einem Tool
# Startpunkt: meine_tools aus Grundlagen erweitern

# 1. berechne_begriffsdichte(text: str, begriff: str) -> str: Begriffsdichte berechnen
# 2. Leere Eingaben als "Fehler: ..."-String zurueckgeben, keine Exception werfen
# 3. Tool zu meine_tools hinzufuegen, Fehlerfall und Normalfall mit .func(...) testen

**Vertiefung**
- Ein drittes Tool mit Mock-Datenbank für Paper-Metadaten bauen.
- Das JSON-Schema per `.args_schema.model_json_schema()` prüfen.
- Ein zweites Tool-Argument mit Standardwert verwenden.

**✅ Erledigt wenn:** Die Mock-Abfrage liefert Paper-Metadaten und das Schema enthält den Standardwert.


In [ ]:
# Vertiefung: Mock-Datenbank und Args-Schema prüfen
import json

PAPER_DB = {
    "rag_survey": {"titel": "Retrieval-Augmented Generation Survey", "thema": "RAG-Architekturen", "jahr": 2023},
    "ragas": {"titel": "RAGAS: Automated Evaluation of Retrieval Augmented Generation", "thema": "Evaluation", "jahr": 2023},
}

# 1. suche_paper_metadaten(paper_id: str, format: str = "kurz") -> dict: Eintrag aus PAPER_DB liefern
# 2. Tool zu meine_tools hinzufuegen, mit .func(...) testen
# 3. Schema per .args_schema.model_json_schema() ausgeben und Standardwert prüfen

<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [KI-Agenten-Architektur](https://editor.p5js.org/ralf.bendig.rb/full/Viso2emNI)
- [KI-Agent](https://editor.p5js.org/ralf.bendig.rb/full/u3Ee0jtFo)

# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [Kursüberblick](https://ralf-42.github.io/Agenten/02-orientierung-entscheidung/kursueberblick.html)
- [Lohnt es sich?](https://ralf-42.github.io/Agenten/02-orientierung-entscheidung/lohnt-es-sich.html)
- [Aufgabenklassen & Lösungswege](https://ralf-42.github.io/Agenten/02-orientierung-entscheidung/aufgabenklassen-und-loesungswege.html)
- [Terminologie](https://ralf-42.github.io/Agenten/02-orientierung-entscheidung/terminologie.html)
- [Meeting- & Research-Briefing-Agent](https://ralf-42.github.io/Agenten/02-orientierung-entscheidung/meeting-research-briefing-leitaufgabe.html)
- [Zuerst lesen](https://ralf-42.github.io/Agenten/zuerst-lesen.html)
- [Lernpfad](https://ralf-42.github.io/Agenten/lernpfad.html)
- [Tool Use & Function Calling](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/tool-use-function-calling.html)
- [Agenten-Architekturen](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/agent-architekturen.html)
- [Weitere Tools](https://ralf-42.github.io/Agenten/05-frameworks/weitere-tools.html)
- [Einsteiger LangChain](https://ralf-42.github.io/Agenten/05-frameworks/einsteiger-langchain.html)
- [Checkliste Agentensystem](https://ralf-42.github.io/Agenten/04-agenten-implementierung/checkliste-agentensystem.html)
